# Trade Gate Bot Team：产品与架构指南

**日期：2026-09-04｜状态：设计指南，不构成 Live 授权**

这份 Notebook 回答四个问题：

1. Grok Bot 的持久 Bot、并行协作、Routine、Memory 与审批设计，哪些值得借鉴？
2. trade-gate 应该有哪些专业角色：情报、交易论点、策略研究、Portfolio、风控、执行、复盘？
3. 多个角色如何并行工作、交接和交流，同时保持“同一账户只有 execd 一个写者”？
4. 前端“小 Bot”如何成为一个可感知、可检查、可接管的值班团队入口？

> 核心结论：前端以一个 **Gate Captain** 作为总入口；后台是多个职责稳定的专业 Bot 和两个受保护的确定性服务。模型可以研究、判断、质疑和提议，但风险裁决与交易所效果不能通过 Bot 间聊天产生。

## 1. Grok Bot 调研摘要

本节只使用 SpaceXAI/xAI 官方材料，访问日期均为 2026-09-04：

- [Introducing Grok Bot](https://x.ai/news/introducing-grok-bot)
- [Designing Grok Bot for a world of persistent agents](https://x.ai/news/designing-grok-bot)
- [Grok Bot overview](https://docs.x.ai/grok-bot/overview)
- [Create and manage Bots](https://docs.x.ai/grok-bot/bots)
- [Message and collaborate](https://docs.x.ai/grok-bot/chat-and-collaboration)
- [Skills and routines](https://docs.x.ai/grok-bot/skills-routines-and-automations)
- [Approvals, security, and privacy](https://docs.x.ai/grok-bot/approvals-security-and-privacy)
- [FAQ](https://docs.x.ai/grok-bot/faq)

### 值得吸收的产品原则

1. **Bot，而不是 Chat，是长期对象。** Bot 有名字、职责、记忆、工具、Routine 和持续状态；Chat 只是与它工作的界面。
2. **角色必须稳定。** 官方建议只有在目标、工具、工作方式、审批边界或周期职责确实不同的时候，才增加新 Bot。
3. **能力共享，记忆按角色隔离。** 多个 Bot 可以共享工具和源系统，但各自保留与职责相关的上下文。重要事实仍要回到权威数据源核对。
4. **并行不等于群聊风暴。** Bot 可以并行、异步消息和群聊交接；但每个阶段应只有一个 owner，过多 handoff 会造成重复工作和噪声。
5. **Routine 是一等对象。** 先把一次任务跑可靠、保存为 Skill，再做定时或事件触发的 Routine；必须定义 stale/no-data 行为、审批点和失败报告。
6. **把工作状态做成 Presence。** 头像表达 idle、thinking、working、waiting、blocked、done；用户需要更多信息时再展开当前动作。
7. **三层监督。** 状态提示 → 侧栏预览 → 需要处理时进入完整工作台。
8. **结构化产物优先。** 对话时间线可以混合文字、事件、表格、卡片、审批对象和可视化。
9. **审批不是装饰。** 财务动作、生产变更等后果性操作需要窄而明确的边界；审批展示目标、范围和具体值。
10. **角色隔离不是安全边界。** Grok Bot 官方明确说明多个 Bot 共享计算机，不能把不同 Bot 当成权限隔离。trade-gate 对应的安全边界必须是 capability、dispatcher、execd、账户写者围栏和数据库状态机。

## 2. trade-gate 当前基线（2026-09-04）

当前不是只有“信息员 + 观察 Agent”，而是：

- **信息员**：定时把行情、funding、OI、情绪和新闻压缩为 `MarketState`。
- **判断模块**：无本币线程时做 scan；有线程时做 review，输出 `NO_TRADE/WATCH/PROPOSE/HOLD/REDUCE/EXIT/INVALIDATE`。
- **主对话 Agent**：查状态、线程、Episode、历史和记忆，可发起扫描/复查、提出交易，但不能直接下单。
- **确定性后台**：触发器、StrategyThread reducer、风险 gate、sizing、Paper/CLI/MCP 执行适配器、订单跟踪与活动流。
- **B7-lite Memory**：SQLite FTS5、结构化 scope、提案审批、召回、supersede/forget、衰减、记忆数字泄漏闸和评测已经落地。

相关实现：`packages/gateway/src/demo/info.ts`、`runtime.ts`、`chat.ts`、`gates.ts`、`memory.ts`；记忆契约见 `docs/demo/memory.md`。

仍缺少的团队能力：

- 持久 Bot registry、每个角色的 profile/memory/routine
- 可审计的 Bot-to-Bot handoff 和 group run
- 账户级 Portfolio 状态、组合约束和组合方案
- 策略实验/优化/晋升工作台
- 独立 Risk Center、告警指纹和 incident 生命周期
- 完整 A2 execd durable execution/reconciliation

> 工作树在本次调研时有其他任务正在并行修改 Binance MCP execution、review metrics 和 UI；这些未提交内容在本指南中只标作 WIP，不算稳定基线。

## 3. 推荐的团队形态

```text
                            Jacky
                              │
                    ┌─────────▼─────────┐
                    │ Gate Captain     │  唯一用户入口/协调者
                    └──────┬──────┬─────┘
             ┌─────────────┘      └──────────────┐
      ┌──────▼──────┐                    ┌──────▼──────┐
      │ Radar 情报员 │                    │ Strategy Lab │
      └──────┬──────┘                    └──────┬──────┘
             │                                  │
      ┌──────▼──────────┐               ┌──────▼──────┐
      │ Thread Manager  │◄─────────────►│ Reviewer     │
      │ 机会+持仓论点   │               │ 评测/复盘    │
      └──────┬──────────┘               └─────────────┘
             │ proposal
      ┌──────▼──────────┐       ┌───────────────────┐
      │ Portfolio Mgr   │◄─────►│ Risk Sentinel     │
      │ 组合影响/分配   │       │ 纯码裁决+模型解释 │
      └──────┬──────────┘       └─────────┬─────────┘
             └──────────┬─────────────────┘
                        │ authorized immutable plan
                 ┌──────▼──────┐
                 │ execd       │  唯一交易所写者
                 │ Executor UI │  只展示状态/回执
                 └─────────────┘
```

界面可以把七个成员都画成团队，但必须用徽章区分：`AI`、`CODE`、`EXEC`。Risk Sentinel 与 Executor 可以有头像和状态，却不能因为有头像就变成自由调用工具的 LLM。

In [ ]:
from dataclasses import dataclass
from typing import Tuple

@dataclass(frozen=True)
class RoleSpec:
    role_id: str
    title: str
    kind: str
    owns: str
    capabilities: Tuple[str, ...]
    memory_scope: str
    approval_boundary: str

ROLES = (
    RoleSpec('gate_captain', 'Gate Captain / 总协调', 'llm_session',
             '用户目标、任务路由、结果汇总、待办与审批收件箱',
             ('state.read', 'bot.delegate', 'intent.propose', 'routine.propose'),
             '用户偏好、沟通方式、团队运行摘要',
             '不能批准自己的提案，不能直接触达交易所'),
    RoleSpec('radar', 'Radar / 信息与发现', 'llm_recipe',
             '市场状态、新闻、候选与 MonitorSpec',
             ('market.read', 'news.read', 'monitor.propose'),
             '信息源质量、候选表现、摘要偏好',
             '只读，不生成订单'),
    RoleSpec('thread_manager', 'Thread Manager / 交易论点', 'llm_recipe',
             '一个 StrategyThread 从 setup 到关闭的论点连续性',
             ('market.read', 'account.read', 'memory.read', 'intent.propose'),
             '按 symbol/strategy/thread 隔离的教训',
             '只提议；数量、杠杆和是否允许由代码决定'),
    RoleSpec('strategy_lab', 'Strategy Lab / 研究与优化', 'llm_session+workers',
             'StrategySpec、实验假设、回测任务和晋升提案',
             ('dataset.read', 'backtest.run', 'paper.register', 'strategy.propose'),
             '研究日志、失败假设、实验结果；不读取 Live 凭证',
             '只能进入 DRAFT/BACKTEST/PAPER，晋升必须人批'),
    RoleSpec('portfolio_manager', 'Portfolio Manager / 组合经理', 'hybrid',
             '总/净/簇敞口、风险预算、资金分配和组合计划',
             ('account.read', 'risk.read', 'portfolio.propose'),
             '组合目标与用户偏好；当前仓位永远现拉',
             '输出 PortfolioPlan，不直接生成交易所效果'),
    RoleSpec('risk_sentinel', 'Risk Sentinel / 风控哨兵', 'deterministic+explainer',
             '实时不变量、gate verdict、incident 与告警',
             ('account.read', 'policy.veto', 'incident.open', 'emergency_reduce.propose'),
             '告警去重与解释模板；实时指标不进长期记忆',
             '代码可以拒绝/收紧，模型永远不能放宽'),
    RoleSpec('reviewer', 'Reviewer / 评测与复盘', 'llm_recipe+deterministic_eval',
             '反方审查、交易复盘、memory/skill/strategy 候选',
             ('episode.read', 'eval.run', 'memory.propose', 'strategy.review'),
             '经批准的 lesson 与评测结论',
             '不能改活跃策略或风险参数，只能提出 diff'),
    RoleSpec('executor', 'Executor / 执行服务', 'protected_service',
             '消费已授权 plan、下单、保护腿、回执与对账',
             ('execution.consume_authorized_plan', 'exchange.write', 'reconcile.write'),
             '无自由文本记忆；只保存六记录、checkpoint 和回执',
             '只接受 plan_hash/account_version/authorization 完整的结构化请求'),
)

for role in ROLES:
    print(f'{role.role_id:18} | {role.kind:28} | {role.title}')

## 4. 职责边界：谁负责什么

### Gate Captain

像 Chief of Staff：接收用户目标、拆任务、指定 owner、并行唤醒角色、等待结果、合成一张决策卡。它不应该自己重新分析所有原始数据，否则专业角色只是昂贵的装饰。

### Radar

把当前信息员与 discovery 合并：维护 MarketState、风险事件、候选漏斗和观察条件。它负责“值得看什么”，不负责“买什么”。

### Thread Manager

继承现在的判断模块，成为一个交易论点的长期 owner。scan 时找 setup，建线程后守护 thesis，收到成交、K 线、情报变化时复查。它只能输出有限状态和 Intent 提案。

### Strategy Lab

专门研究和优化策略。它可以并行生成假设、写 StrategySpec、调用回测、比较版本、注册 Paper 策略，但永远不能把研究结果直接推入 Live。

### Portfolio Manager

站在账户层看问题：BTC 与 ETH 同向仓是否属于同一个风险簇？总敞口、净敞口、保证金、回撤和资金费是否允许新增风险？它输出组合影响与一组有依赖顺序的 `PortfolioPlan`，而不是直接下多笔单。

### Risk Sentinel

实时部分是纯代码。LLM 只把 gate verdict 和 incident 翻译成人话。它拥有 veto 和收紧能力，没有放宽能力；Critical alert 可以抢占普通 Bot 工作。

### Reviewer

同时做 proposal 反方审查与交易后复盘。它可以指出证据缺口、拥挤交易、反向情景和过拟合，但不能修改经济字段。结论通过 Memory/Skill/Strategy proposal 进入审批。

### Executor

它在 UI 里可以像一个专业成员，但实现上是 `execd` 和受限解释器。它不参与开放式群聊，只消费不可变计划并发布结构化状态。

In [ ]:
def validate_role_boundaries(roles):
    errors = []
    writers = []
    for role in roles:
        caps = set(role.capabilities)
        if 'exchange.write' in caps:
            writers.append(role.role_id)
        if role.kind.startswith('llm') and 'exchange.write' in caps:
            errors.append(f'{role.role_id}: LLM 不得拥有 exchange.write')
        if role.role_id == 'risk_sentinel' and 'policy.veto' not in caps:
            errors.append('risk_sentinel 缺少 veto')
        if role.role_id == 'gate_captain' and 'exchange.write' in caps:
            errors.append('Gate Captain 不得执行订单')
    if writers != ['executor']:
        errors.append(f'exchange writer 必须且只能是 executor，实际为 {writers}')
    return errors

boundary_errors = validate_role_boundaries(ROLES)
assert not boundary_errors, boundary_errors
print('Role boundary check: PASS')

## 5. 并行模型：不是所有东西都同时跑

推荐六类 lane：

| Lane | 并发方式 | 原因 |
|---|---|---|
| `intel` | 单飞，新的请求合并 | 防止重复拉相同全市场信息 |
| `judgment:<symbol>` | 不同 symbol 并行，同 symbol 串行 | 防止同一论点产生冲突判断 |
| `research:<strategy>` | 2–4 个受限 worker 并行 | 参数、市场区间、反向测试可并行 |
| `portfolio:<account_version>` | 同一账户版本单飞 | 所有组合建议必须基于同一快照 |
| `risk` | 纯码实时、最高优先级 | 不等待模型、不受预算耗尽影响 |
| `execution:<account>` | 每账户唯一写 lane | 满足单写者与 durable execution |

可并行的典型提案审查：

```text
Thread Manager 提出 setup
          ├─ Portfolio Manager：计算组合影响
          ├─ Reviewer：寻找反方证据/重复风险
          └─ Risk Sentinel：纯码跑 gate preview
                       ↓
Gate Captain 等三者到齐或超时 → 合成给 Jacky 的卡片
                       ↓
用户批准 → 执行前重闸 → execd 单写 lane
```

并行分析可以 fail-partial：Reviewer 超时不应伪装成通过；卡片应显示“反方审查未完成”。执行则必须 fail-closed。

In [ ]:
EVENT_ROUTES = {
    'market_state.updated': ('radar', 'gate_captain'),
    'trigger.hit': ('thread_manager',),
    'proposal.created': ('portfolio_manager', 'risk_sentinel', 'reviewer'),
    'order.filled': ('thread_manager', 'risk_sentinel', 'gate_captain'),
    'protection.missing': ('risk_sentinel', 'executor', 'gate_captain'),
    'thread.closed': ('reviewer', 'strategy_lab', 'portfolio_manager'),
    'strategy.experiment.finished': ('reviewer', 'strategy_lab'),
    'incident.critical': ('risk_sentinel', 'gate_captain'),
}

def lane_for(event_name, payload):
    if event_name.startswith('trigger.') or event_name.startswith('order.'):
        return f"judgment:{payload['symbol']}"
    if event_name.startswith('strategy.'):
        return f"research:{payload['strategy_id']}"
    if event_name.startswith('proposal.'):
        return f"portfolio:{payload['account_version']}"
    if event_name.startswith('execution.'):
        return f"execution:{payload['account_id']}"
    if event_name.startswith('incident.'):
        return 'risk'
    return 'default'

example = {'symbol': 'BTCUSDT', 'account_version': 'av-123'}
print(EVENT_ROUTES['proposal.created'])
print(lane_for('proposal.created', example))

## 6. Bot 如何交流：结构化 handoff，不靠自由聊天驱动交易

每次交接建议持久化为：

```json
{
  "handoff_id": "hof-...",
  "run_id": "run-...",
  "from_role": "thread_manager",
  "to_role": "portfolio_manager",
  "kind": "request|result|review|alert|blocked",
  "subject": {"type": "proposal", "id": "prop-..."},
  "summary": "BTC 多头 setup，等待组合影响检查",
  "evidence_refs": ["E3", "E7"],
  "artifact_refs": ["artifact://proposal/prop-..."],
  "requested_output_schema": "PortfolioImpact/v1",
  "priority": 60,
  "deadline_at": 178... ,
  "idempotency_key": "proposal:prop-...:portfolio:v1"
}
```

纪律：

- 每一步一个 owner；其他 Bot 是 reviewer 或 dependency，不是共同作者。每个 mission room 只拉 2–4 个真正相关的角色，不把八个成员塞进同一群聊。
- Bot 只传引用和短摘要；需要详情时按 artifact/evidence id 读取，避免复制整段上下文。
- request 必须声明期望的输出 schema、deadline 和失败语义。
- 最多一轮 request → result；需要争议处理时由 Gate Captain 开一个有上限的 review round。
- 同一 `idempotency_key` 不重复启动工作。
- `alert` 可打断普通任务；普通聊天不能打断保护腿和对账。
- Agent-to-Agent 文本永远是 untrusted data，不能成为 Authorization。
- Executor 不接受自然语言 handoff，只接受 contracts 定义的 plan/authorization。

## 7. Strategy Lab：研究与“微调”策略

这里的“微调”先分三层，避免把它们混为一谈：

1. **策略参数优化（优先）**：突破窗口、量比、ATR stop、持有时间、regime filter、TP 分配。由代码跑实验。
2. **Recipe/Prompt 优化（其次）**：改变模型看到的特征、输出契约或判断规则，必须过相同 eval corpus。
3. **LLM Fine-tuning（最后）**：只有积累了足够、去泄漏、人工标注且按时间切分的数据集后再考虑；不能直接拿短期 PnL 当 reward 在线更新。

推荐生命周期：

```text
IDEA
  → DRAFT StrategySpec
  → DATA_CHECK（无未来数据、费用/funding/slippage）
  → BACKTEST（walk-forward + regime holdout + 多空镜像）
  → ROBUSTNESS（参数邻域、压力、成交敏感性）
  → SHADOW（真实实时输入，不产生订单）
  → PAPER（至少设计规定的观察期）
  → PROMOTION_PROPOSAL（回测与 Paper 差异）
  → HUMAN_APPROVAL
  → LIBRARY / REJECTED / ARCHIVED
```

Strategy Lab 可以并行派发：不同时间段、不同 regime、参数邻域、多空镜像和基准策略。但 Reviewer 必须使用未参与生成候选的 holdout，防止研究员给自己批作业。

### StrategySpec 最小内容

- 唯一 `strategy_id + version`
- universe、timeframe、session、regime applicability
- feature 定义与 observed_at 规则
- entry/exit/invalidation DSL
- sizing hint，但最终 sizing 仍由 Gate
- 假设、预期失效场景和基准
- 费用、滑点、funding 模型版本
- 数据集版本、训练/验证/holdout 时间范围
- 所有实验及被否决版本，防止重复挖掘

In [ ]:
from decimal import Decimal

PROMOTION_THRESHOLDS = {
    'schema_validity': Decimal('0.99'),
    'evidence_validity': Decimal('0.98'),
    'stale_refusal': Decimal('1.00'),
    'unauthorized_action_rate_max': Decimal('0'),
    'paper_days_min': Decimal('28'),
}

def promotion_check(metrics):
    checks = {
        'schema_validity': Decimal(metrics['schema_validity']) >= PROMOTION_THRESHOLDS['schema_validity'],
        'evidence_validity': Decimal(metrics['evidence_validity']) >= PROMOTION_THRESHOLDS['evidence_validity'],
        'stale_refusal': Decimal(metrics['stale_refusal']) >= PROMOTION_THRESHOLDS['stale_refusal'],
        'unauthorized_action_rate': Decimal(metrics['unauthorized_action_rate']) <= PROMOTION_THRESHOLDS['unauthorized_action_rate_max'],
        'paper_days': Decimal(metrics['paper_days']) >= PROMOTION_THRESHOLDS['paper_days_min'],
        'holdout_positive': metrics['holdout_positive'] is True,
        'parameter_stable': metrics['parameter_stable'] is True,
    }
    return {'eligible_for_human_review': all(checks.values()), 'checks': checks}

candidate = {
    'schema_validity': '0.995', 'evidence_validity': '0.991', 'stale_refusal': '1.00',
    'unauthorized_action_rate': '0', 'paper_days': '31',
    'holdout_positive': True, 'parameter_stable': False,
}
promotion_check(candidate)

## 8. Portfolio Manager：从单币判断升级到账户级决策

Portfolio Manager 每次都从 `account.truth` 读取新鲜数据，不能依赖长期记忆中的仓位数字。建议生成一个不可变 `PortfolioSnapshot`：

- equity、available margin、margin ratio
- 每个账户、symbol、方向、策略和风险簇的 notional
- gross exposure、net exposure、有效 leverage
- 单币/单策略/相关簇集中度
- realized/unrealized PnL、daily loss、drawdown
- funding carry 与未来 funding 时间
- open orders 和潜在成交后的 projected exposure
- snapshot components 的 observed_at、completeness、consistency 和 `account_version`

它处理三类任务：

1. **Proposal impact**：如果这笔单成交，组合会变成什么？
2. **Periodic health**：是否过度集中、相关方向重复、保证金恶化或资金费拖累？
3. **Portfolio plan**：提出减仓、资金预留、再平衡或停止新增风险的建议。

多腿再平衡不是原子交易。计划必须先执行降低风险的腿，再重新读取账户并决定是否允许增加风险；不能假设失败后一定可以按原价回滚。

In [ ]:
from collections import defaultdict
from dataclasses import dataclass

@dataclass(frozen=True)
class Position:
    symbol: str
    side: str
    notional: str
    cluster: str

def portfolio_exposure(equity_text, positions):
    equity = Decimal(equity_text)
    gross = Decimal('0')
    net = Decimal('0')
    by_cluster = defaultdict(lambda: Decimal('0'))
    by_symbol = defaultdict(lambda: Decimal('0'))
    for p in positions:
        n = Decimal(p.notional)
        signed = n if p.side == 'long' else -n
        gross += abs(n)
        net += signed
        by_cluster[p.cluster] += signed
        by_symbol[p.symbol] += signed
    concentration = max((abs(v) for v in by_symbol.values()), default=Decimal('0')) / equity
    return {
        'gross_notional': str(gross),
        'net_notional': str(net),
        'effective_leverage': str((gross / equity).quantize(Decimal('0.0001'))),
        'largest_symbol_fraction': str(concentration.quantize(Decimal('0.0001'))),
        'cluster_net': {k: str(v) for k, v in sorted(by_cluster.items())},
    }

positions = [
    Position('BTCUSDT', 'long', '2400.00', 'crypto_major'),
    Position('ETHUSDT', 'long', '1800.00', 'crypto_major'),
    Position('SOLUSDT', 'short', '600.00', 'crypto_beta'),
]
portfolio_exposure('10000.00', positions)

## 9. Risk Sentinel：专业风控与告警

Risk Sentinel 的心脏应该是纯函数和高频哨兵，不是大模型。建议至少覆盖：

### 市场与组合风险

- gross/net exposure、单币/策略/相关簇集中度
- 有效 leverage、margin ratio、清算距离
- 日亏、连续亏损、峰值回撤和盈利回吐
- funding 极端、流动性/价差、价格偏离
- open order 成交后的 projected exposure

### 执行与系统风险

- `execution_unknown`
- protection missing / 数量不覆盖
- 账户快照 inconsistent/unavailable/stale
- OAuth、MCP session、tools snapshot、API key 权限异常
- user stream 断流、时钟漂移、429 ban、单写者冲突
- 外部订单或 Binance UI 手工活动

### 告警对象

每条 `RiskAlert` 应包含 severity、fingerprint、首次/最后观察时间、当前值、阈值、source refs、受影响账户/线程、自动动作、需要人工做什么和 resolved_at。条件实质变化后 fingerprint 才变化，避免每 10 秒重复轰炸。

建议动作层级：

- `INFO`：记录，不打扰
- `WARN`：Bot 变黄，进入收件箱
- `HIGH`：停止新增风险，通知用户
- `CRITICAL`：抢占普通任务，进入 `FLATTEN_ONLY/HALT_ALL`；只允许预先定义的 EmergencyReduce

## 10. Executor：可以有人格，但不能有自由意志

Executor 的 UI 可以像一名严谨的执行员，状态包括 validating、queued、submitting、awaiting_ack、protecting、reconciling、blocked、done；但协议必须保持机器化：

1. 只接收 `ExecutableOrderPlan + Authorization`。
2. 校验 `plan_hash`、`account_version`、authorization TTL 和 policy version。
3. 调用前持久化 `clientOrderId`。
4. 每账户唯一 durable writer lane。
5. 超时进入 `execution_unknown`，只能按订单身份对账，不能盲目重发。
6. 入场成交后确认原生保护腿；保护失败按事故策略降险。
7. 发布 ExchangeOrder、Fill、PositionEffect 和 trace，不发布“我觉得成交了”。
8. Bot-to-Bot 消息、聊天文本和 Memory 都不能成为 Authorization。

前端归因建议：

- “Thread Manager 提出 BTC 多头方案”
- “Portfolio Manager：相关簇敞口将升至 …”
- “Risk Sentinel：17/18 项通过，1 项阻断”
- “Jacky 于 10:42 批准 plan hash …”
- “Executor 已提交；交易所回执 …；保护腿已确认”

不要统一写成“Gate Bot 下了一单”，否则审计关系会被人格化文案掩盖。

## 11. 五条核心 Workflow

### 11.1 情报与机会

`数据 → Radar → MarketState → 代码触发器 → Thread Manager → WATCH/PROPOSE`

WATCH 应持久化为 `WatchCandidate`，包含重新唤醒条件和 TTL，而不是只留一句话。

### 11.2 Proposal Council

`PROPOSE → Portfolio impact ∥ Reviewer countercase ∥ Risk preview → Gate Captain 汇总 → 用户审批`

### 11.3 持仓守护

`成交/部分成交/K线/信息变化 → Thread Manager review ∥ Risk Sentinel → HOLD/REDUCE/EXIT`

保护腿和止损触发不等待 Agent；Agent 只管理论点和解释异常。

### 11.4 Strategy Lab

`研究目标 → 并行实验 → Reviewer holdout → Shadow → Paper → PromotionProposal → 人批`

### 11.5 Learning Loop

`平仓 → 确定性结果 → Reviewer 复盘 → Memory/Strategy diff → Eval → 人批 → 新版本`

任何学习产物都不能在同一条交易尚未结算时反向修改它的规则。

## 12. 前端：一个小 Bot，背后是一支团队

### 全局 Gate Bot

常驻右下角，状态由 SSE/持久事件驱动：idle、collecting、thinking、reviewing、waiting approval、executing、guarding、blocked、halted、done。模型不能直接设置动画状态。

### 三层查看

1. **Glance**：头像 + 一句话 + 数字徽章，例如“守护 2 个线程｜1 项待批”。
2. **Preview drawer**：当前工作、Bot roster、最近 handoff、风险告警、审批和下次 Routine。
3. **Takeover/workspace**：打开 Portfolio、Risk、Execution、Strategy Lab 或具体 thread/incident。

### Bot roster

每个成员卡显示头像、类型徽章、状态、当前任务、最近产物、下一 Routine、最后心跳和是否需要你。默认折叠空闲成员，避免把用户变成调度员。

### Group room / Mission timeline

不要复制普通群聊。用一个异构时间线混合：

- Bot 消息
- handoff 事件
- Market Pulse / WatchCandidate
- Proposal / PortfolioImpact / GateVerdict
- ApprovalCard / ExecutionReceipt
- RiskAlert / Incident
- StrategyExperiment / PromotionProposal
- Memory diff / Review report

用户可以 `@Radar`、`@Portfolio`，但默认发给 Gate Captain，由它路由。

### 推荐导航

- Gate（总入口/收件箱）
- Intel（Radar 工作台）
- Threads（机会与持仓）
- Portfolio（账户与组合）
- Risk Center（敞口、告警、incident）
- Execution（Intent、回执、对账）
- Strategy Lab（实验、Paper、晋升）
- Reviews & Memory
- Routines

可爱程度应随风险下降：空闲/研究可以有性格；等待审批保持克制；Critical/HALT 只显示明确红色状态、原因和动作。

## 13. 建议增加的持久对象

| 对象 | 用途 |
|---|---|
| `bot_profiles` | role、description、model pin、capabilities、memory scope |
| `bot_runs` | 一次有界工作、owner、状态、预算、deadline |
| `bot_handoffs` | 可审计异步 request/result/review/alert |
| `artifacts` | 所有结构化产物的版本和 URI |
| `routines` / `routine_runs` | schedule/event trigger、owner、失败史 |
| `watch_candidates` | WATCH 条件、TTL、再次唤醒规则 |
| `portfolio_snapshots` | account_version 对应的组合视图 |
| `portfolio_plans` | 多腿建议及依赖顺序，不直接代表授权 |
| `risk_alerts` / `incidents` | 告警指纹、严重度、生命周期 |
| `strategy_versions` | StrategySpec 不可变版本 |
| `experiments` | dataset、参数、代码版本、指标与结果 |
| `promotion_proposals` | Backtest/Shadow/Paper 对比与人工决策 |

`bot_handoffs` 和 `artifacts` 应存引用，不复制账户快照。账户和行情继续由权威源按 `observed_at/account_version` 获取。

## 14. 分阶段落地建议

### Phase 1：把现有系统包装成 Gate Bot（低风险）

- 全局小 Bot + deterministic presence 状态机
- Preview drawer：activity、queue、thread、intent、risk 状态
- 现有信息员、判断、对话、Memory 映射成角色
- 结构化卡片和 deep link，不新增交易权限

### Phase 2：可审计的多 Bot 协作

- `bot_profiles/bot_runs/bot_handoffs/artifacts`
- Gate Captain dispatcher
- symbol/account/research/execution lanes
- Proposal Council 并行 fan-out/fan-in
- role-scoped memory；共享事实只通过 artifact/evidence

### Phase 3：Portfolio + Risk Center

- 先做确定性 exposure engine 和 projected exposure
- 风险 alert fingerprint、incident、Telegram/WebUI 通知
- Portfolio Manager 只生成组合建议
- 账户真相不完整时 fail-closed

### Phase 4：Strategy Lab

- StrategySpec、dataset version、backtest runner
- walk-forward/holdout/robustness/基准
- Shadow/Paper 注册与晋升卡
- 先优化规则参数，后做 prompt；暂不做在线 LLM fine-tune

### Phase 5：Execution hardening

- 按权威设计完成 A2 execd 六记录、durable queue、单写者、reconciler 和 kill tests
- Executor 只消费批准且重闸通过的 plan
- Paper 策略就绪与执行就绪分开报告

### Phase 6：监督金丝雀

只有策略就绪、执行就绪、额度拍板和 Binance 身份/授权问题均解决后进入；不因 Bot 团队看起来聪明就提前开放 Live。

## 15. 第一版验收标准

### 产品体验

- 用户 3 秒内知道系统在做什么、是否需要自己、当前最大风险是什么。
- 点开一次即可从 Bot 状态追到 run → handoff → evidence → artifact → intent。
- 所有审批都显示提出者、反方审查、组合影响、gate verdict 和具体经济字段。
- 空闲 Bot 不制造动态；相同告警不重复轰炸。

### 协作正确性

- 每个 run 恰好一个 owner。
- 同一 symbol 判断串行，不同 symbol 可并行。
- 同一 account_version 的 Portfolio 分析可复放。
- Bot-to-Bot 消息有 schema、deadline、幂等键和引用。
- 没有无限 delegation loop；超过 handoff/turn/token 上限即 blocked。

### 安全

- 任意 LLM role 都拿不到凭证和 `exchange.write`。
- 只有 execd 是账户效果写者。
- Risk Sentinel 可以拒绝或收紧，模型不能放宽。
- critical incident、execution_unknown、protection missing 会抢占普通工作并冻结新增敞口。
- Memory、聊天和 handoff 不能提供实时价格、仓位、权限或 Authorization。

### 研究质量

- 每个 Strategy version 绑定 dataset/code/cost model。
- 有时间 holdout、regime holdout、基准、多空镜像和参数邻域。
- 失败实验保留，防止重复数据挖掘。
- Promotion 只是进入人工审查，不是自动上线。

## 16. 建议现在拍板的产品问题

1. 前端唯一主角是否叫 **Gate Captain/Gate**，其他角色作为团队成员？
2. Thread Manager 是按 symbol 长期存在，还是按每条 StrategyThread 临时产生？建议：角色长期存在，thread 是它负责的任务对象。
3. Portfolio Manager 第一版只看 Agentic 子账户，还是合并主账户只读视图？建议先子账户，主账户只做上下文和资金边界。
4. Reviewer 是否对每个 PROPOSE 都运行，还是只对高风险/低置信/相关簇拥挤运行？建议 Paper 期全跑，之后基于规则抽样。
5. Strategy Lab 第一批支持参数网格、贝叶斯优化，还是只做人工提出的有限候选？建议有限候选 + walk-forward，先保证可解释和防过拟合。
6. 小 Bot 的人格强度：专业值班员、轻度拟人，还是明显卡通？建议专业值班员，研究/空闲有性格，风险/执行克制。
7. Bot 间 handoff 是否全部对用户可见？建议全部可审计，但主时间线只显示摘要，详情按需展开。

---

**一句话产品定义**：Trade Gate 是一支 24/7 并行工作的交易值班团队——Radar 找变化，Thread Manager 维护论点，Strategy Lab 研究方法，Portfolio Manager 看全局，Risk Sentinel 守边界，Executor 只执行已授权计划，Gate Captain 把需要 Jacky 判断的事情压缩到一个收件箱。